# 📅 2026-04-12 (일) 개발 노트 : Django 백엔드 코어 구축 및 4,190개 데이터 마이그레이션

## 🎯 오늘의 목표
- [x] Django 프로젝트(`django_core`) 초기 세팅 및 앱(`games`, `users`) 구조 설계
- [x] Docker PostgreSQL(pgvector)와 Django 연동
- [x] GPT-5.4로 추출한 4,190개 스팀 게임 데이터 DB 벌크 적재(`load_games.py`)
- [x] 신규/기존 스팀 게임 확장을 위한 Few-Shot 저비용 분석 파이프라인 설계(`analyze_new_games.py`)
- [x] `.gitignore` 세팅 및 프로젝트 명세서 문서화 완료

## 🛠 진행 상황 및 핵심 기록
- **백엔드 구조 셋업:** `config` (설정), `games` (핵심 로직), `users` (유저 관리)로 MSA 확장을 고려한 폴더 구조 생성
- **DB 연동 (`settings.py`):** `psycopg2-binary`를 활용해 Docker 컨테이너 DB(`hidden_gem_db`)와 연결 성공
- **데이터 적재 스크립트:** `Game`과 `GameMetric` 모델을 `OneToOneField`로 분리하여 설계. 데이터 삽입 시 성능을 위해 `bulk_create` 및 `Transaction` 적용.
- **파이프라인 아키텍처:** 신규 게임 수집 시 API 비용을 1/240로 줄이는 전략(GPT-4o-mini + 5.4 기반 Few-Shot) 코어 스크립트 작성 완료.

## 🚨 트러블슈팅 (문제 및 해결)
- **문제 1:** DB 연결 시 `FATAL: role "postgres" does not exist` 에러 발생.
  - **원인:** Docker 컨테이너 생성 시 기본 유저를 `postgres`가 아닌 다른 이름으로 설정함.
  - **해결:** `docker exec hidden_gem_vectordb env | grep POSTGRES` 명령어로 환경변수 확인 후 진짜 유저명(`juntae`)을 찾아 적용.
- **문제 2:** 로컬 서버 띄운 후 `/kimjuntae` 접속 시 404 에러 발생.
  - **원인:** Django `urls.py`에 등록되지 않은 경로로 접속 시도. 서버 에러가 아닌 정상적인 라우팅 방어 작동.
  - **해결:** `http://localhost:8000/admin` 정확한 관리자 경로로 이동하여 해결.
- **문제 3:** Admin 패널에서 분석 방법 뱃지가 `batch_gpt4`로 짤려서 표기됨 (5.4 하드코딩 누락).
  - **원인:** 스크립트에 4.1 버전이 하드코딩되어 들어갔고, Admin 표시 글자 수가 제한(10자)되어 있었음.
  - **해결:** `models.py`의 `choices`와 `admin.py`의 `method_badge` 수정 후, `python manage.py shell`에서 `update()` ORM으로 4,190개 데이터를 일괄 'gpt5.4_batch'로 수정.
- **문제 4:** 데이터 업데이트 적재 중 `Connection refused (0x0000274D/10061)` 에러.
  - **원인:** 컴퓨터 재부팅/유휴 상태로 인해 Docker PostgreSQL 컨테이너가 내려간 상태에서 5432 포트 접근.
  - **해결:** `docker start hidden_gem_vectordb`로 DB를 깨운 후 스크립트 재실행.

## 💡 인사이트 및 다음 할 일
- **배운 점:** 1. Django Admin의 사기성: 코드 몇 줄로 52개 지표 필터링, 데이터 시각화(뱃지)가 가능한 강력한 백오피스가 만들어짐.
  2. 대용량 처리의 차이: `bulk_create`는 10초 만에 끝나지만, DB에 존재하는 기존 데이터를 일일이 찾아 덮어쓰는 `update` 로직은 쿼리 수(약 16,700번)가 많아 시간이 더 소요됨을 체감.
  3. Git 보안 관리: 프로젝트 설계도(`.ipynb`)나 로컬 데이터(`data/`)는 `.gitignore`에 명시해 원격 저장소 오염을 막는 것이 실무 표준임을 배움.
- **다음 할 일:** [인터넷 설치 후] FastAPI 연동(고속 검색 엔진 구축) 및 10만 개 스팀 Back-catalog 싹쓸이 파이프라인(Few-shot Nightly Batch) 가동.